# IMPROVE Addis Analog Audit

This notebook answers the IMPROVE analog question directly:

**Does the IMPROVE network contain samples that reproduce the Addis HIPS offset / optical-floor behavior, and what can IMPROVE rule out about the Addis anomaly?**

## Working answer up front

The current IMPROVE export does **not** support high EC surface loading alone as a general explanation for the Addis HIPS offset. Once IMPROVE samples are restricted to the Addis/ETAD EC surface-loading regime, most rows follow a fairly normal fAbs-vs-EC relationship. Addis-like high-fAbs rows are rare, episodic, and often carry smoke-like or RT-saturation signatures.

The strongest IMPROVE result is therefore a boundary condition:

> IMPROVE can produce smoke/event rows with high absorption, but it does not show a widespread persistent Addis-style optical floor across normal loading-matched samples.

## What this notebook can and cannot test

The IMPROVE workbooks used here contain final chemistry, fAbs, flow duration, and 635 nm reflectance/transmittance ratio fields. They do **not** expose the full raw HIPS blank audit needed to reproduce the Addis SOP checks (`a0`, `a1`, raw blank rows, blank `t + r`, blank `tau`).

So this notebook can test:

- network-scale fAbs-vs-EC behavior,
- Addis/ETAD loading-regime overlap,
- smoke/event candidate structure,
- RT saturation / low-transmittance behavior in exported ratio fields.

It cannot directly test:

- field-blank zeroing,
- lot-level blank regression coefficients,
- exact SOP tau recalculation from raw `T`, `R`, `a0`, `a1`.

## Authoritative references used here

- [IMPROVE SOP #276: Optical Absorption Analysis of PM2.5 Samples](https://aqrc.ucdavis.edu/sites/g/files/dgvnsk1671/files/inline-files/IMPROVE_SOP_276_v5.5_Optical_Absorption_Analysis_of_PM2.5_Samples.pdf)
- [Ren et al. 2025, *Nature Communications*: Black carbon emissions generally underestimated in the global south as revealed by globally distributed measurements](https://www.nature.com/articles/s41467-025-62468-5)
- [UC Davis / IMPROVE gray literature: Teflon material classification](https://improvevisibility.org/docs/Publications/GrayLit/040_Teflon_Material_Classification/040_teflon_material_classification_09212020.pdf)

Outputs are written under `research/improve_hips_offset/output/improve_addis_analog_audit/`. Plots are shown inline when the notebook is run.

## 0. Setup

This follows the repo guidance in `AGENTS.md`: use `/opt/anaconda3/bin/python3.13`, import reusable logic from `research/ftir_hips_chem/scripts`, use the shared plotting style, and write all generated outputs under a notebook-specific output directory.

In [ ]:
%matplotlib inline

from pathlib import Path
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore")

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "research" / "ftir_hips_chem").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root from current working directory.")

REPO_ROOT = find_repo_root()
RESEARCH_DIR = REPO_ROOT / "research" / "ftir_hips_chem"
THIS_DIR = REPO_ROOT / "research" / "improve_hips_offset"
OUT_DIR = THIS_DIR / "output" / "improve_addis_analog_audit"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCRIPTS_DIR = RESEARCH_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from config import SITES, FILTER_DATA_PATH, MAC_VALUE, MIN_EC_THRESHOLD
from outliers import apply_exclusion_flags, apply_threshold_flags
from plotting import PlotConfig, apply_default_style

apply_default_style()
PlotConfig.set(sites="all", layout="individual", show_stats=True, show_1to1=True)

DEFAULT_DRIVE_ROOT = (
    Path.home()
    / "Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data"
)
DRIVE_ROOT = Path(os.environ.get("AETHMODULAR_MAIA_DATA_ROOT", DEFAULT_DRIVE_ROOT)).expanduser().resolve()
IMPROVE_DIR = Path(os.environ.get("AETHMODULAR_IMPROVE_DIR", DRIVE_ROOT / "Improve")).expanduser().resolve()

SITE_ORDER = ["Beijing", "Delhi", "JPL", "Addis_Ababa"]
SITE_CODE_TO_NAME = {cfg["code"]: site for site, cfg in SITES.items()}
SITE_COLORS = {site: SITES[site]["color"] for site in SITE_ORDER}
MIN_EC = MIN_EC_THRESHOLD
PRIMARY_IMPROVE_AREA_CM2 = 3.5
DEPOSIT_AREA_FALLBACK_CM2 = 3.53
NA_VALUES = [-999, "-999", -999.0, "-999.0"]
KEY_COLS = ["Dataset", "SiteCode", "POC", "Date", "AuxID"]
TRANS_SATURATION_RATIO = 0.01
REF_SATURATION_RATIO = 0.01
OC_EC_SMOKE_MIN = 5.0
RANDOM_SEED = 42
STRICT_CURRENT_EXPORT_CHECKS = True

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)

for required_path in [FILTER_DATA_PATH, IMPROVE_DIR]:
    assert required_path.exists(), f"Missing required input: {required_path}"

print(f"Repo root: {REPO_ROOT}")
print(f"Research dir: {RESEARCH_DIR}")
print(f"IMPROVE dir: {IMPROVE_DIR}")
print(f"Output dir: {OUT_DIR}")

## 1. Rebuild the Addis/ETAD reference regime from SPARTAN source data

The IMPROVE analog test needs a reference target. This cell rebuilds the SPARTAN four-site HIPS-vs-FTIR baseline directly from `unified_filter_dataset.pkl`, applies the repo exclusion registry, and extracts the Addis/ETAD fAbs and EC surface-loading regime.

This mirrors the Addis audit pattern: source data → canonical filtered table → baseline regressions → target regime.

In [ ]:
def regression_stats(df: pd.DataFrame, x_col: str, y_col: str) -> dict:
    pair = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    pair = pair[(pair[x_col] > 0) & (pair[y_col] > 0)]
    if len(pair) < 3 or pair[x_col].nunique() < 2:
        return {
            "n": len(pair),
            "Slope": np.nan,
            "Intercept": np.nan,
            "R2": np.nan,
            "origin_mac": np.nan,
        }
    x = pair[x_col].to_numpy(float)
    y = pair[y_col].to_numpy(float)
    slope, intercept, r, p, se = stats.linregress(x, y)
    origin_mac = float(np.dot(x, y) / np.dot(x, x))
    return {
        "n": len(pair),
        "Slope": slope,
        "Intercept": intercept,
        "R2": r ** 2,
        "origin_mac": origin_mac,
    }


df_long = pd.read_pickle(FILTER_DATA_PATH)
df_long["SampleDate"] = pd.to_datetime(df_long["SampleDate"], errors="coerce")
df_long["SiteName"] = df_long["Site"].map(SITE_CODE_TO_NAME)

pivot_params = [
    "EC_ftir", "HIPS_Fabs", "HIPS_tau", "HIPS_T1", "HIPS_R1",
    "HIPS_t", "HIPS_r", "HIPS_Intercept", "HIPS_Slope",
    "ChemSpec_OC_PM2.5", "ChemSpec_Filter_PM2.5_mass",
]
filter_wide = (
    df_long[df_long["Parameter"].isin(pivot_params)]
    .pivot_table(
        index=["Site", "SiteName", "FilterId", "SampleDate", "FilterType", "LotId", "DepositArea_cm2", "Volume_m3"],
        columns="Parameter",
        values="Concentration",
        aggfunc="first",
    )
    .reset_index()
)
filter_wide.columns.name = None
filter_wide["HIPS_BC"] = filter_wide["HIPS_Fabs"] / MAC_VALUE

ec_mass = (
    df_long[df_long["Parameter"].eq("EC_ftir")][["FilterId", "MassLoading_ug"]]
    .drop_duplicates("FilterId")
    .rename(columns={"MassLoading_ug": "EC_loading_ug"})
)
filter_wide = filter_wide.merge(ec_mass, on="FilterId", how="left")
filter_wide["EC_loading_ug_cm2"] = filter_wide["EC_loading_ug"] / filter_wide["DepositArea_cm2"].fillna(DEPOSIT_AREA_FALLBACK_CM2)

clean_frames = []
for site_name in SITE_ORDER:
    site_code = SITES[site_name]["code"]
    site_df = filter_wide[(filter_wide["Site"] == site_code) & (filter_wide["FilterType"] == "PM2.5")].copy()
    site_df["aeth_bc"] = pd.NA
    site_df["filter_ec"] = site_df["EC_ftir"] * 1000.0
    site_df["date"] = site_df["SampleDate"]
    site_df["filter_id"] = site_df["FilterId"]
    site_df = apply_exclusion_flags(site_df, site_name)
    site_df = apply_threshold_flags(site_df, site_name)
    clean_df = site_df[
        (~site_df["is_excluded"])
        & (~site_df["is_outlier"])
        & site_df["EC_ftir"].notna()
        & site_df["HIPS_BC"].notna()
        & (site_df["EC_ftir"] >= MIN_EC)
    ].copy()
    clean_frames.append(clean_df)

clean_pm25 = pd.concat(clean_frames, ignore_index=True)

baseline_summary = (
    clean_pm25.groupby("SiteName")
    .apply(lambda g: pd.Series(regression_stats(g, "EC_ftir", "HIPS_BC")))
    .loc[SITE_ORDER]
)
tau_summary = (
    clean_pm25.groupby("SiteName")
    .apply(lambda g: pd.Series(regression_stats(g, "EC_ftir", "HIPS_tau")))
    .loc[SITE_ORDER]
)

etad = clean_pm25[clean_pm25["SiteName"].eq("Addis_Ababa")].copy()
ETAD_FABS_MIN = float(etad["HIPS_Fabs"].min())
ETAD_FABS_MAX = float(etad["HIPS_Fabs"].max())
ETAD_FABS_MEDIAN = float(etad["HIPS_Fabs"].median())
ETAD_LOADING_P05 = float(etad["EC_loading_ug_cm2"].quantile(0.05))
ETAD_LOADING_P25 = float(etad["EC_loading_ug_cm2"].quantile(0.25))
ETAD_LOADING_MEDIAN = float(etad["EC_loading_ug_cm2"].median())
ETAD_LOADING_P75 = float(etad["EC_loading_ug_cm2"].quantile(0.75))
ETAD_LOADING_P95 = float(etad["EC_loading_ug_cm2"].quantile(0.95))

spartan_ref_rows = []
for site in SITE_ORDER:
    g = clean_pm25[clean_pm25["SiteName"].eq(site)]
    spartan_ref_rows.append({
        "SiteName": site,
        "n_filtered": len(g),
        "HIPS_Fabs_median": g["HIPS_Fabs"].median(),
        "HIPS_Fabs_min": g["HIPS_Fabs"].min(),
        "HIPS_Fabs_max": g["HIPS_Fabs"].max(),
        "EC_ftir_median": g["EC_ftir"].median(),
        "EC_loading_ug_cm2_p05": g["EC_loading_ug_cm2"].quantile(0.05),
        "EC_loading_ug_cm2_median": g["EC_loading_ug_cm2"].median(),
        "EC_loading_ug_cm2_p95": g["EC_loading_ug_cm2"].quantile(0.95),
        "HIPS_tau_median": g["HIPS_tau"].median(),
        "HIPS_tau_min": g["HIPS_tau"].min(),
        "HIPS_tau_max": g["HIPS_tau"].max(),
    })
spartan_ref = pd.DataFrame(spartan_ref_rows)
spartan_ref.to_csv(OUT_DIR / "spartan_rebuilt_reference.csv", index=False)

expected_baseline = {
    "Beijing": {"n": 144, "Slope": 0.603, "Intercept": 0.539, "R2": 0.663},
    "Delhi": {"n": 57, "Slope": 0.712, "Intercept": 0.933, "R2": 0.777},
    "JPL": {"n": 60, "Slope": 0.842, "Intercept": 0.057, "R2": 0.295},
    "Addis_Ababa": {"n": 190, "Slope": 0.402, "Intercept": 2.832, "R2": 0.764},
}
if STRICT_CURRENT_EXPORT_CHECKS:
    for site, expected in expected_baseline.items():
        actual = baseline_summary.loc[site]
        assert int(actual["n"]) == expected["n"], f"{site}: expected n={expected['n']}, got {actual['n']}"
        for field in ["Slope", "Intercept", "R2"]:
            assert np.isclose(actual[field], expected[field], atol=0.01), (
                f"{site}: {field} drifted ({actual[field]} vs {expected[field]})"
            )

print("Filtered SPARTAN HIPS_BC vs FTIR EC baseline:")
display(baseline_summary.round(4))
print("Filtered SPARTAN HIPS_tau vs FTIR EC baseline:")
display(tau_summary.round(4))
print("Addis/ETAD reference regime:")
display(spartan_ref.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8))

ax = axes[0]
for site in SITE_ORDER:
    g = clean_pm25[clean_pm25["SiteName"].eq(site)]
    ax.scatter(
        g["EC_ftir"], g["HIPS_BC"], s=36, alpha=0.72,
        color=SITE_COLORS[site], edgecolor="white", linewidth=0.4, label=site
    )
    fit = baseline_summary.loc[site]
    x_line = np.linspace(g["EC_ftir"].min(), g["EC_ftir"].max(), 120)
    ax.plot(x_line, fit["Slope"] * x_line + fit["Intercept"], color=SITE_COLORS[site], lw=1.8)
ax.plot([0, clean_pm25["EC_ftir"].max()], [0, clean_pm25["EC_ftir"].max()], color="0.25", ls=":", lw=1.2, label="1:1")
ax.set_title("SPARTAN reference: HIPS BC vs FTIR EC")
ax.set_xlabel("FTIR EC (ug/m3)")
ax.set_ylabel("HIPS BC = HIPS_Fabs / MAC (ug/m3)")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

ax = axes[1]
for site in SITE_ORDER:
    g = clean_pm25[clean_pm25["SiteName"].eq(site)]
    ax.scatter(
        g["EC_ftir"], g["HIPS_tau"], s=36, alpha=0.72,
        color=SITE_COLORS[site], edgecolor="white", linewidth=0.4, label=site
    )
    fit = tau_summary.loc[site]
    x_line = np.linspace(g["EC_ftir"].min(), g["EC_ftir"].max(), 120)
    ax.plot(x_line, fit["Slope"] * x_line + fit["Intercept"], color=SITE_COLORS[site], lw=1.8)
ax.axhline(0, color="0.25", ls=":", lw=1.1)
ax.set_title("SPARTAN reference: HIPS tau vs FTIR EC")
ax.set_xlabel("FTIR EC (ug/m3)")
ax.set_ylabel("HIPS tau")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

fig.suptitle("Addis is the reference anomaly: positive floor and compressed slope", y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(OUT_DIR / "spartan_reference_baseline.png", dpi=240, bbox_inches="tight")
plt.show()

### Interpretation

The SPARTAN baseline is the target for the IMPROVE comparison:

- Addis has a large positive HIPS_BC intercept and a positive HIPS_tau intercept.
- The Addis fAbs regime is roughly `28-86 Mm^-1`.
- The Addis EC surface-loading p05-p95 regime is roughly `4.7-19.6 ug/cm2`.

IMPROVE analogs should therefore satisfy two conditions at once: they should overlap Addis-like EC surface loading **and** Addis-like fAbs. Rows with only high fAbs but extreme mass/smoke or RT saturation are useful event candidates, but not automatically root-cause analogs.

## 2. Load and join the IMPROVE chemistry and RT exports

This uses the two new IMPROVE workbooks:

- chemistry/fAbs/mass/flow workbook,
- 635 nm reflectance/transmittance ratio workbook.

`-999` sentinel values are converted to missing values before all calculations.

In [ ]:
def find_excel_with_columns(directory: Path, required_cols: list[str]) -> Path:
    required_cols = set(required_cols)
    for path in sorted(directory.glob("*.xlsx")):
        try:
            xl = pd.ExcelFile(path)
            if "Data" not in xl.sheet_names:
                continue
            cols = pd.read_excel(xl, sheet_name="Data", nrows=0).columns.astype(str).str.strip().tolist()
            if required_cols.issubset(cols):
                return path
        except Exception:
            continue
    raise FileNotFoundError(f"No Excel workbook in {directory} contains {sorted(required_cols)}")


def standardize_improve_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = out.columns.astype(str).str.strip()
    if "Date" in out.columns:
        out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    for col in out.columns:
        if col.endswith("_Val") or col in {"POC", "AuxID"}:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def ratio_safe(num, den):
    num = pd.to_numeric(num, errors="coerce")
    den = pd.to_numeric(den, errors="coerce")
    return pd.Series(
        np.where((den > 0) & np.isfinite(den) & np.isfinite(num), num / den, np.nan),
        index=num.index,
    )


def duplicate_summary(df: pd.DataFrame, key_cols=KEY_COLS) -> dict:
    dup_mask = df.duplicated(key_cols, keep=False)
    return {
        "duplicate_rows": int(dup_mask.sum()),
        "duplicate_key_groups": int(df.loc[dup_mask, key_cols].drop_duplicates().shape[0]),
    }


chem_path = find_excel_with_columns(IMPROVE_DIR, ["SiteCode", "Date", "ECf_Val", "OCf_Val", "fAbs_Val"])
rt_path = find_excel_with_columns(IMPROVE_DIR, ["SiteCode", "Date", "RefF_635_Val", "TransF_635_Val"])
print(f"Chemistry workbook: {chem_path.name}")
print(f"RT workbook: {rt_path.name}")

chem_params = pd.read_excel(chem_path, sheet_name="Parameters", na_values=NA_VALUES)
rt_params = pd.read_excel(rt_path, sheet_name="Parameters", na_values=NA_VALUES)
sites = pd.read_excel(chem_path, sheet_name="Sites", na_values=NA_VALUES)
overview = pd.read_excel(chem_path, sheet_name="Overview", na_values=NA_VALUES)

chem = standardize_improve_frame(pd.read_excel(chem_path, sheet_name="Data", na_values=NA_VALUES))
rt = standardize_improve_frame(pd.read_excel(rt_path, sheet_name="Data", na_values=NA_VALUES))

inventory = pd.DataFrame([
    {
        "table": "chemistry",
        "rows": len(chem),
        "sites": chem["SiteCode"].nunique(),
        "date_min": chem["Date"].min(),
        "date_max": chem["Date"].max(),
        **duplicate_summary(chem),
    },
    {
        "table": "rt",
        "rows": len(rt),
        "sites": rt["SiteCode"].nunique(),
        "date_min": rt["Date"].min(),
        "date_max": rt["Date"].max(),
        **duplicate_summary(rt),
    },
])

value_cols = [c for c in chem.columns if c.endswith("_Val")]
rt_value_cols = [c for c in rt.columns if c.endswith("_Val")]
missingness = pd.concat([
    chem[value_cols].isna().mean().mul(100).rename("missing_pct").reset_index().assign(table="chemistry"),
    rt[rt_value_cols].isna().mean().mul(100).rename("missing_pct").reset_index().assign(table="rt"),
], ignore_index=True).rename(columns={"index": "column"}).loc[:, ["table", "column", "missing_pct"]]

print("IMPROVE workbook inventory:")
display(inventory)
print("Value-column missingness (%):")
display(missingness.sort_values(["table", "missing_pct"], ascending=[True, False]).round(2))
print("Chemistry parameters:")
display(chem_params)
print("RT parameters:")
display(rt_params)

In [ ]:
chem_dedup = chem.sort_values(KEY_COLS).drop_duplicates(KEY_COLS, keep="last")
rt_dedup = rt.sort_values(KEY_COLS).drop_duplicates(KEY_COLS, keep="last")
joined = chem_dedup.merge(rt_dedup, on=KEY_COLS, how="left", suffixes=("", "_rt"), validate="one_to_one")

site_keep = [
    c for c in [
        "Code", "Site", "Country", "State", "County", "Latitude", "Longitude",
        "Elevation", "Sponsor", "ProgramKey",
    ]
    if c in sites.columns
]
site_meta = sites[site_keep].drop_duplicates("Code").rename(columns={"Code": "SiteCode", "Site": "SiteName"})
joined = joined.merge(site_meta, on="SiteCode", how="left")

joined["valid_ec_fabs"] = (
    joined["ECf_Val"].notna()
    & joined["fAbs_Val"].notna()
    & (joined["ECf_Val"] > 0)
    & (joined["fAbs_Val"] > 0)
)
joined["flow_duration_valid"] = (
    joined["FlowRate_Val"].notna()
    & joined["SampDur_Val"].notna()
    & (joined["FlowRate_Val"] > 0)
    & (joined["SampDur_Val"] > 0)
)
joined["volume_m3"] = np.where(
    joined["flow_duration_valid"],
    joined["FlowRate_Val"] * joined["SampDur_Val"] / 1000.0,
    np.nan,
)
joined["EC_loading_ug"] = joined["ECf_Val"] * joined["volume_m3"]
joined["EC_loading_ug_cm2_area_3p5"] = joined["EC_loading_ug"] / PRIMARY_IMPROVE_AREA_CM2
joined["loading_available"] = joined["EC_loading_ug_cm2_area_3p5"].notna()
joined["fAbs_per_EC"] = ratio_safe(joined["fAbs_Val"], joined["ECf_Val"])
joined["OC_EC"] = ratio_safe(joined["OCf_Val"], joined["ECf_Val"])
joined["FE_EC"] = ratio_safe(joined["FEf_Val"], joined["ECf_Val"])
joined["SOIL_EC"] = ratio_safe(joined["SOILf_Val"], joined["ECf_Val"])

rt_core = [
    "RefF_635_Val", "TransF_635_Val", "RefI_635_Val", "TransI_635_Val",
    "RefM_635_Val", "TransM_635_Val",
]
joined["rt_available"] = joined[rt_core].notna().all(axis=1)
joined["trans_min_over_initial"] = ratio_safe(joined["TransM_635_Val"], joined["TransI_635_Val"])
joined["trans_final_over_initial"] = ratio_safe(joined["TransF_635_Val"], joined["TransI_635_Val"])
joined["ref_min_over_initial"] = ratio_safe(joined["RefM_635_Val"], joined["RefI_635_Val"])
joined["ref_final_over_initial"] = ratio_safe(joined["RefF_635_Val"], joined["RefI_635_Val"])
joined["trans_od_proxy"] = -np.log(joined["trans_min_over_initial"].clip(lower=1e-9))
joined["trans_zero_or_saturated"] = (
    joined["rt_available"]
    & (
        (joined["TransM_635_Val"] <= 0)
        | (joined["TransI_635_Val"] <= 0)
        | (joined["trans_min_over_initial"] <= TRANS_SATURATION_RATIO)
    )
)
joined["ref_zero_or_saturated"] = (
    joined["rt_available"]
    & (
        (joined["RefM_635_Val"] <= 0)
        | (joined["RefI_635_Val"] <= 0)
        | (joined["ref_min_over_initial"] <= REF_SATURATION_RATIO)
    )
)
joined["rt_saturated"] = joined["trans_zero_or_saturated"] | joined["ref_zero_or_saturated"]

western_states = {"CA", "OR", "WA", "MT", "ID", "NV", "AZ", "NM", "CO", "UT", "WY"}
joined["western_us"] = joined["State"].isin(western_states)
joined["warm_season"] = joined["Date"].dt.month.between(6, 10)
joined["year_month"] = joined["Date"].dt.to_period("M").astype(str)

valid = joined[joined["valid_ec_fabs"]].copy()
qc = joined[joined["valid_ec_fabs"] & joined["rt_available"] & joined["loading_available"]].copy()

joined.to_csv(OUT_DIR / "improve_joined_audit_qc.csv", index=False)
try:
    joined.to_parquet(OUT_DIR / "improve_joined_audit_qc.parquet", index=False)
except Exception as exc:
    print(f"Parquet cache skipped: {type(exc).__name__}: {exc}")

join_summary = pd.DataFrame([
    {"metric": "joined rows", "value": len(joined)},
    {"metric": "joined sites", "value": joined["SiteCode"].nunique()},
    {"metric": "valid positive EC/fAbs rows", "value": len(valid)},
    {"metric": "RT + EC loading available rows", "value": len(qc)},
    {"metric": "rows with RT saturation flag", "value": int(qc["rt_saturated"].sum())},
    {"metric": "RT saturation pct within QC", "value": qc["rt_saturated"].mean() * 100},
])
display(join_summary)
display(joined.head())

## 3. What IMPROVE can and cannot say about blank zeroing

The Addis audit could test field blanks directly because the SPARTAN raw HIPS export contained blank filters, raw `T1`/`R1`, coefficients, `t`, `r`, and `tau`.

The IMPROVE export here is different: it contains fAbs and RT ratio fields, but not blank rows or SOP coefficient fields. This section makes that limitation explicit so the interpretation does not overclaim.

In [ ]:
joined_cols = pd.Index(joined.columns)
blank_like_cols = [c for c in joined_cols if "blank" in c.lower()]
coefficient_like_cols = [c for c in joined_cols if c.lower() in {"a0", "a1", "intercept", "slope"} or "coef" in c.lower()]
raw_tau_like_cols = [c for c in joined_cols if c.lower() in {"tau", "t", "r"} or c.lower().endswith("_tau")]

capability_table = pd.DataFrame([
    {
        "audit_question": "Can we test field blank t+r≈1?",
        "available_in_improve_export": False,
        "reason": "No FilterType/blank rows and no raw blank t/r/tau fields in the exported Data sheet.",
    },
    {
        "audit_question": "Can we recalculate SOP tau = ln((a0+a1R)/T)?",
        "available_in_improve_export": False,
        "reason": "No a0/a1 coefficient fields and no raw T/R count fields equivalent to the SPARTAN HIPS export.",
    },
    {
        "audit_question": "Can we identify RT saturation / low transmittance regimes?",
        "available_in_improve_export": True,
        "reason": "RT export includes Ref/Trans initial/final/min ratio fields at 635 nm.",
    },
    {
        "audit_question": "Can we test network-scale fAbs vs EC behavior?",
        "available_in_improve_export": True,
        "reason": "Chemistry export includes ECf and fAbs, plus flow and sample duration for EC surface loading.",
    },
])
print("Blank-like columns found:", blank_like_cols)
print("Coefficient-like columns found:", coefficient_like_cols)
print("Raw tau/t/r-like columns found:", raw_tau_like_cols)
display(capability_table)

assert "FilterType" not in joined.columns
assert len(coefficient_like_cols) == 0

### Interpretation

IMPROVE can support or weaken *analog* hypotheses, but it cannot directly repeat the Addis field-blank audit from this export alone.

If we need root-cause confirmation from IMPROVE, the missing files are:

- per-filter raw HIPS `T` and `R` counts,
- field blank rows,
- PTFE lot IDs,
- lot-level blank-regression coefficients (`a0`, `a1`),
- any HIPS processing flags for saturation or out-of-range verification.

## 4. Addis/ETAD comparability in IMPROVE

This is the direct analog test:

1. Restrict IMPROVE to rows with valid EC, fAbs, RT, and EC surface loading.
2. Ask how many rows overlap the Addis/ETAD EC surface-loading range.
3. Ask how many also overlap the Addis/ETAD fAbs range.
4. Compare the fAbs-vs-EC fits across the full, loading-matched, and fAbs+loading-matched groups.

In [ ]:
joined["etad_loading_p05p95"] = joined["EC_loading_ug_cm2_area_3p5"].between(
    ETAD_LOADING_P05, ETAD_LOADING_P95, inclusive="both"
)
joined["etad_loading_iqr"] = joined["EC_loading_ug_cm2_area_3p5"].between(
    ETAD_LOADING_P25, ETAD_LOADING_P75, inclusive="both"
)
joined["etad_loading_or_higher"] = joined["EC_loading_ug_cm2_area_3p5"] >= ETAD_LOADING_P05
joined["etad_fabs_range"] = joined["fAbs_Val"].between(ETAD_FABS_MIN, ETAD_FABS_MAX, inclusive="both")
joined["etad_loading_and_fabs"] = joined["etad_loading_p05p95"] & joined["etad_fabs_range"]

qc = joined[joined["valid_ec_fabs"] & joined["rt_available"] & joined["loading_available"]].copy()
valid = joined[joined["valid_ec_fabs"]].copy()
loading_matched = qc[qc["etad_loading_p05p95"]].copy()
loading_iqr = qc[qc["etad_loading_iqr"]].copy()
fabs_matched = qc[qc["etad_fabs_range"]].copy()
both_matched = qc[qc["etad_loading_and_fabs"]].copy()
high_tail_loading = loading_matched[loading_matched["fAbs_Val"] >= 70].copy()

improve_groups = {
    "valid_positive_ec_fabs": valid,
    "rt_and_loading_available": qc,
    "ETAD_loading_p05p95": loading_matched,
    "ETAD_loading_IQR": loading_iqr,
    "ETAD_fAbs_only_with_qc": fabs_matched,
    "ETAD_fAbs_plus_ETAD_loading": both_matched,
    "fAbs_ge70_plus_ETAD_loading": high_tail_loading,
}

group_rows = []
fit_rows = []
for name, g in improve_groups.items():
    group_rows.append({
        "group": name,
        "n": len(g),
        "sites": g["SiteCode"].nunique() if len(g) else 0,
        "fAbs_median": g["fAbs_Val"].median() if len(g) else np.nan,
        "fAbs_max": g["fAbs_Val"].max() if len(g) else np.nan,
        "EC_median": g["ECf_Val"].median() if len(g) else np.nan,
        "EC_loading_median": g["EC_loading_ug_cm2_area_3p5"].median() if len(g) else np.nan,
        "rt_saturated_pct": g["rt_saturated"].mean() * 100 if len(g) else np.nan,
    })
    fit = regression_stats(g, "ECf_Val", "fAbs_Val")
    fit_rows.append({
        "group": name,
        **fit,
        "site_count": g["SiteCode"].nunique() if len(g) else 0,
        "fAbs_median": g["fAbs_Val"].median() if len(g) else np.nan,
        "EC_median": g["ECf_Val"].median() if len(g) else np.nan,
    })

group_summary = pd.DataFrame(group_rows)
fit_summary = pd.DataFrame(fit_rows)
group_summary.to_csv(OUT_DIR / "improve_analog_group_counts.csv", index=False)
fit_summary.to_csv(OUT_DIR / "improve_analog_fit_summary.csv", index=False)

expected_counts = {
    "valid_positive_ec_fabs": 379697,
    "rt_and_loading_available": 145200,
    "ETAD_loading_p05p95": 6946,
    "ETAD_fAbs_plus_ETAD_loading": 24,
    "fAbs_ge70_plus_ETAD_loading": 1,
}
if STRICT_CURRENT_EXPORT_CHECKS:
    actual_counts = dict(zip(group_summary["group"], group_summary["n"]))
    for group, expected in expected_counts.items():
        assert actual_counts[group] == expected, f"{group}: expected {expected}, got {actual_counts[group]}"

print("IMPROVE analog group counts:")
display(group_summary.round(4))
print("IMPROVE fAbs-vs-EC fits:")
display(fit_summary.round(4))

In [ ]:
fig = plt.figure(figsize=(17, 6))
gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.35, 1.15], wspace=0.32)

ax = fig.add_subplot(gs[0, 0])
bars = group_summary.set_index("group")["n"].plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xscale("log")
ax.set_xlabel("Rows (log scale)")
ax.set_title("IMPROVE analog-group sizes")
ax.grid(axis="x", alpha=0.25)

ax = fig.add_subplot(gs[0, 1])
base = qc[(qc["ECf_Val"] <= 8) & (qc["fAbs_Val"] <= 120)].copy()
hb = ax.hexbin(
    base["ECf_Val"], base["fAbs_Val"], gridsize=65, extent=(0, 8, 0, 120),
    mincnt=1, bins="log", cmap="Greys", alpha=0.72
)
fig.colorbar(hb, ax=ax, pad=0.01, label="log10 count")
ax.scatter(
    loading_matched["ECf_Val"], loading_matched["fAbs_Val"],
    s=24, color="#4C72B0", alpha=0.45, label=f"ETAD loading p05-p95 (n={len(loading_matched):,})"
)
ax.scatter(
    both_matched["ECf_Val"], both_matched["fAbs_Val"],
    s=76, color="#D55E00", edgecolor="black", linewidth=0.45, alpha=0.95,
    label=f"ETAD fAbs + loading (n={len(both_matched):,})"
)
if len(high_tail_loading):
    ax.scatter(
        high_tail_loading["ECf_Val"], high_tail_loading["fAbs_Val"],
        s=120, color="#C0392B", edgecolor="black", linewidth=0.65, alpha=0.98,
        label=f"fAbs>=70 + loading (n={len(high_tail_loading):,})"
    )
ax.axhspan(ETAD_FABS_MIN, ETAD_FABS_MAX, color="#F39C12", alpha=0.14, label="Addis/ETAD fAbs range")
x_line = np.linspace(0, 8, 200)
ax.plot(x_line, MAC_VALUE * x_line, color="0.15", lw=1.2, ls=":", label=f"MAC={MAC_VALUE:g}")
ax.set_xlim(0, 8)
ax.set_ylim(0, 120)
ax.set_xlabel("IMPROVE ECf (ug/m3)")
ax.set_ylabel("IMPROVE fAbs (Mm-1)")
ax.set_title("IMPROVE fAbs vs EC with Addis/ETAD overlays")
ax.legend(loc="upper right", fontsize=7.5, frameon=True)
ax.grid(alpha=0.25)

ax = fig.add_subplot(gs[0, 2])
plot_fit = fit_summary.set_index("group").loc[
    ["valid_positive_ec_fabs", "rt_and_loading_available", "ETAD_loading_p05p95", "ETAD_fAbs_plus_ETAD_loading"],
    ["Slope", "Intercept", "origin_mac"],
]
plot_fit[["Slope", "origin_mac"]].plot(kind="bar", ax=ax, width=0.75)
ax.axhline(MAC_VALUE, color="0.2", ls=":", lw=1.2, label=f"MAC={MAC_VALUE:g}")
ax.set_title("Slope/MAC proxy by IMPROVE subset")
ax.set_ylabel("fAbs per EC")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8)

fig.suptitle("IMPROVE does not show a widespread Addis-style high-offset regime", y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(OUT_DIR / "improve_addis_analog_overview.png", dpi=240, bbox_inches="tight")
plt.show()

### Interpretation

The analog filter is severe:

- `6,946` IMPROVE rows match Addis/ETAD EC surface loading.
- Only `24` rows match both Addis/ETAD loading and Addis/ETAD fAbs.
- Only `1` row is both Addis/ETAD-loading-matched and `fAbs >= 70`.

This weakens the idea that high EC surface loading alone creates a stable Addis-like optical floor. If that were a general network behavior, the ETAD-loading-matched IMPROVE group should show a broad, persistent high intercept or many high-tail analogs. It does not.

## 5. Smoke/event screen

This section asks whether the rare IMPROVE high-fAbs analogs look like episodic smoke/high-organic-mass events rather than a persistent instrument floor.

The score intentionally matches the earlier IMPROVE QC notebook so counts remain comparable.

In [ ]:
smoke_thresholds = {
    "fAbs_p99": qc["fAbs_Val"].quantile(0.99),
    "fAbs_p995": qc["fAbs_Val"].quantile(0.995),
    "OCf_p95": qc["OCf_Val"].quantile(0.95),
    "MF_p90": qc["MF_Val"].quantile(0.90),
    "OC_EC_p90": qc["OC_EC"].quantile(0.90),
    "trans_od_proxy_p90": qc["trans_od_proxy"].quantile(0.90),
    "ETAD_loading_p05": ETAD_LOADING_P05,
}

char = qc.copy()
char["char_high_loading"] = char["EC_loading_ug_cm2_area_3p5"] >= smoke_thresholds["ETAD_loading_p05"]
char["char_high_fAbs_p99"] = char["fAbs_Val"] >= smoke_thresholds["fAbs_p99"]
char["char_very_high_fAbs_p995"] = char["fAbs_Val"] >= smoke_thresholds["fAbs_p995"]
char["char_high_OC"] = char["OCf_Val"] >= smoke_thresholds["OCf_p95"]
char["char_organic_rich"] = char["OC_EC"] >= OC_EC_SMOKE_MIN
char["char_extreme_OC_EC"] = char["OC_EC"] >= smoke_thresholds["OC_EC_p90"]
char["char_high_MF"] = char["MF_Val"] >= smoke_thresholds["MF_p90"]
char["char_rt_optical_loading"] = char["trans_od_proxy"] >= smoke_thresholds["trans_od_proxy_p90"]

char["smoke_like_score"] = (
    2 * char["char_high_fAbs_p99"].astype(int)
    + 1 * char["char_very_high_fAbs_p995"].astype(int)
    + 2 * char["char_high_OC"].astype(int)
    + 1 * char["char_organic_rich"].astype(int)
    + 1 * char["char_extreme_OC_EC"].astype(int)
    + 1 * char["char_high_MF"].astype(int)
    + 1 * char["char_high_loading"].astype(int)
    + 1 * char["char_rt_optical_loading"].astype(int)
)
char["event_context"] = np.select(
    [
        char["western_us"] & char["warm_season"],
        char["western_us"],
        char["warm_season"],
    ],
    ["western warm-season", "western other-season", "nonwestern warm-season"],
    default="nonwestern other-season",
)

p99_loading_ocec = char[char["char_high_fAbs_p99"] & char["char_high_loading"] & char["char_organic_rich"]].copy()
score_ge8 = char[char["smoke_like_score"] >= 8].copy()
score_ge9 = char[char["smoke_like_score"] >= 9].copy()

expected_candidate_counts = {
    "p99_loading_ocec": 335,
    "score_ge8": 407,
    "score_ge9": 144,
}
if STRICT_CURRENT_EXPORT_CHECKS:
    assert len(p99_loading_ocec) == expected_candidate_counts["p99_loading_ocec"]
    assert len(score_ge8) == expected_candidate_counts["score_ge8"]
    assert len(score_ge9) == expected_candidate_counts["score_ge9"]

candidate_groups = {
    "QC all RT + loading": char,
    "fAbs >= QC p99": char[char["char_high_fAbs_p99"]],
    "fAbs >= QC p99 + ETAD-low loading + OC/EC >= 5": p99_loading_ocec,
    "smoke_like_score >= 8": score_ge8,
    "smoke_like_score >= 9": score_ge9,
}
candidate_rows = []
for name, g in candidate_groups.items():
    candidate_rows.append({
        "group": name,
        "n": len(g),
        "sites": g["SiteCode"].nunique() if len(g) else 0,
        "western_us_pct": g["western_us"].mean() * 100 if len(g) else np.nan,
        "warm_season_pct": g["warm_season"].mean() * 100 if len(g) else np.nan,
        "rt_saturated_pct": g["rt_saturated"].mean() * 100 if len(g) else np.nan,
        "fAbs_median": g["fAbs_Val"].median() if len(g) else np.nan,
        "EC_loading_median": g["EC_loading_ug_cm2_area_3p5"].median() if len(g) else np.nan,
        "OC_EC_median": g["OC_EC"].median() if len(g) else np.nan,
        "MF_median": g["MF_Val"].median() if len(g) else np.nan,
    })
candidate_summary = pd.DataFrame(candidate_rows)
candidate_summary.to_csv(OUT_DIR / "improve_smoke_candidate_summary.csv", index=False)

candidate_cols = [
    "SiteCode", "SiteName", "State", "Date", "ECf_Val", "OCf_Val", "fAbs_Val",
    "EC_loading_ug_cm2_area_3p5", "OC_EC", "MF_Val", "trans_min_over_initial",
    "trans_od_proxy", "rt_saturated", "western_us", "warm_season", "event_context", "smoke_like_score",
]
smoke_candidates = score_ge8.sort_values(["smoke_like_score", "fAbs_Val"], ascending=[False, False])[candidate_cols].copy()
smoke_candidates.to_csv(OUT_DIR / "improve_smoke_event_candidates.csv", index=False)

display(pd.DataFrame([{"threshold": k, "value": v} for k, v in smoke_thresholds.items()]).round(4))
display(candidate_summary.round(4))
display(smoke_candidates.head(30))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.6), gridspec_kw={"width_ratios": [1.0, 1.05, 1.25]})

ax = axes[0]
context_counts = (
    score_ge8.groupby("event_context")
    .size()
    .reindex(["western warm-season", "western other-season", "nonwestern warm-season", "nonwestern other-season"])
    .fillna(0)
)
ax.bar(context_counts.index, context_counts.values, color=["#D55E00", "#E69F00", "#56B4E9", "#999999"])
ax.set_title("Smoke-like candidates by context")
ax.set_ylabel("Rows")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)

ax = axes[1]
top_sites = score_ge8["SiteCode"].value_counts().head(15).sort_values()
ax.barh(top_sites.index, top_sites.values, color="#4C72B0")
ax.set_title("Top candidate sites")
ax.set_xlabel("Score >= 8 rows")
ax.grid(axis="x", alpha=0.25)

ax = axes[2]
top_site_order = score_ge8["SiteCode"].value_counts().head(15).index.tolist()
heat = (
    score_ge8[score_ge8["SiteCode"].isin(top_site_order)]
    .pivot_table(index="SiteCode", columns="year_month", values="smoke_like_score", aggfunc="size", fill_value=0)
    .reindex(top_site_order)
)
if heat.empty:
    ax.text(0.5, 0.5, "No candidates", ha="center", va="center")
    ax.axis("off")
else:
    im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd")
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels(heat.index)
    xticks = np.arange(0, len(heat.columns), max(1, len(heat.columns) // 7))
    ax.set_xticks(xticks)
    ax.set_xticklabels([heat.columns[i] for i in xticks], rotation=45, ha="right")
    ax.set_title("Candidate site-month clustering")
    fig.colorbar(im, ax=ax, pad=0.01, label="Rows")

fig.suptitle("High-fAbs IMPROVE rows are episodic and site/month clustered", y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(OUT_DIR / "improve_smoke_candidate_context.png", dpi=240, bbox_inches="tight")
plt.show()

site_month_counts = (
    score_ge8
    .groupby(["SiteCode", "SiteName", "State", "year_month"], dropna=False)
    .size()
    .reset_index(name="n_candidates")
    .sort_values(["n_candidates", "SiteCode", "year_month"], ascending=[False, True, True])
)
site_month_counts.to_csv(OUT_DIR / "improve_smoke_candidate_site_month_counts.csv", index=False)
display(site_month_counts.head(30))

### Interpretation

The smoke-like candidates are not random network noise:

- the strict p99 + loading + OC/EC screen keeps `335` rows,
- the score ≥ 8 screen keeps `407` rows,
- many candidates are western and/or warm-season,
- candidates cluster by site-month.

That makes them useful for event follow-up, but it also means they should not be treated as a stable Addis-like instrument floor without row-level fire/smoke confirmation.

## 6. RT optical-regime and saturation audit

The IMPROVE RT workbook does not expose raw SOP `T`/`R`, but it does expose initial, final, and minimum transmittance/reflectance ratio fields at 635 nm. We use those as an optical-regime screen.

Rows with zero or near-zero `TransM / TransI` are handled as **saturation class rows**, not as precise optical-depth measurements.

In [ ]:
saturation_candidates = qc[qc["rt_saturated"]].sort_values(["fAbs_Val", "EC_loading_ug_cm2_area_3p5"], ascending=[False, False]).copy()
saturation_candidates.to_csv(OUT_DIR / "improve_rt_saturation_candidates.csv", index=False)

rt_groups = {
    "RT + loading available": qc,
    "ETAD loading p05-p95": loading_matched,
    "ETAD fAbs + loading": both_matched,
    "smoke_like_score >= 8": score_ge8,
    "RT saturated": saturation_candidates,
    "RT not saturated": qc[~qc["rt_saturated"]],
}

rt_rows = []
for name, g in rt_groups.items():
    rt_rows.append({
        "group": name,
        "n": len(g),
        "sites": g["SiteCode"].nunique() if len(g) else 0,
        "trans_min_over_initial_median": g["trans_min_over_initial"].median() if len(g) else np.nan,
        "ref_min_over_initial_median": g["ref_min_over_initial"].median() if len(g) else np.nan,
        "trans_od_proxy_median": g["trans_od_proxy"].median() if len(g) else np.nan,
        "fAbs_median": g["fAbs_Val"].median() if len(g) else np.nan,
        "EC_loading_median": g["EC_loading_ug_cm2_area_3p5"].median() if len(g) else np.nan,
        "rt_saturated_pct": g["rt_saturated"].mean() * 100 if len(g) else np.nan,
    })
rt_summary = pd.DataFrame(rt_rows)
rt_summary.to_csv(OUT_DIR / "improve_rt_saturation_summary.csv", index=False)
display(rt_summary.round(4))
saturation_display_cols = [c for c in candidate_cols if c in saturation_candidates.columns]
display(saturation_candidates[saturation_display_cols].head(25))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.6))

ax = axes[0]
bg = loading_matched.dropna(subset=["RefM_635_Val", "TransM_635_Val"])
hb = ax.hexbin(
    bg["RefM_635_Val"], bg["TransM_635_Val"],
    gridsize=55, mincnt=1, bins="log", cmap="Greys", alpha=0.72
)
fig.colorbar(hb, ax=ax, pad=0.01, label="log10 count")
sat_lm = loading_matched[loading_matched["rt_saturated"]]
ax.scatter(
    sat_lm["RefM_635_Val"], sat_lm["TransM_635_Val"],
    s=58, color="#C0392B", edgecolor="black", linewidth=0.35, alpha=0.88,
    label=f"Saturated within ETAD loading (n={len(sat_lm):,})"
)
ax.set_xlabel("RefM_635")
ax.set_ylabel("TransM_635")
ax.set_title("RT geometry proxy")
ax.legend(fontsize=8)

ax = axes[1]
nonsat = qc[~qc["rt_saturated"]].dropna(subset=["trans_min_over_initial", "fAbs_Val"])
sat = qc[qc["rt_saturated"]].dropna(subset=["trans_min_over_initial", "fAbs_Val"])
ax.scatter(nonsat["trans_min_over_initial"], nonsat["fAbs_Val"], s=10, color="0.55", alpha=0.18, label=f"Not saturated (n={len(nonsat):,})")
ax.scatter(sat["trans_min_over_initial"].clip(lower=1e-6), sat["fAbs_Val"], s=24, color="#C0392B", alpha=0.55, label=f"Saturated (n={len(sat):,})")
ax.axvline(TRANS_SATURATION_RATIO, color="#C0392B", ls="--", lw=1.2)
ax.set_xscale("log")
ax.set_xlabel("TransM_635 / TransI_635")
ax.set_ylabel("fAbs (Mm-1)")
ax.set_title("fAbs vs transmittance ratio")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

ax = axes[2]
score_ge8.boxplot(column="fAbs_Val", by="rt_saturated", ax=ax, grid=False)
ax.set_title("Smoke-like fAbs by RT saturation")
ax.set_xlabel("RT saturated")
ax.set_ylabel("fAbs (Mm-1)")
fig.suptitle("")
ax.grid(axis="y", alpha=0.25)

fig.suptitle("RT saturation separates many high-score smoke/event rows", y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(OUT_DIR / "improve_rt_saturation_audit.png", dpi=240, bbox_inches="tight")
plt.show()

### Interpretation

The RT fields support a cautious interpretation:

- RT saturation is uncommon across all QC rows, but much more common among high-score smoke-like candidates.
- A saturated row can be a real heavy-smoke event, an optical-regime edge case, or both.
- Saturated rows should not be used as clean Addis analogs without raw HIPS/QC logs.

## 7. Regression sensitivity: does IMPROVE show an Addis-like floor?

Addis-like behavior would look like a persistent positive intercept and/or a compressed dynamic range in a clean loading-matched group.

We compare OLS, origin-through-zero MAC, and sampled Theil-Sen robust fits across:

- all valid IMPROVE,
- RT + loading rows,
- ETAD-loading-matched rows,
- ETAD fAbs + loading rows,
- smoke-like candidates,
- RT saturated / not saturated rows.

In [ ]:
def robust_regression_stats(df, x_col="ECf_Val", y_col="fAbs_Val"):
    pair = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    pair = pair[(pair[x_col] > 0) & (pair[y_col] > 0)]
    base = regression_stats(pair, x_col, y_col)
    if len(pair) < 3:
        base.update({"theilsen_slope": np.nan, "theilsen_intercept": np.nan, "theilsen_n": len(pair)})
        return base
    ts_pair = pair
    if len(ts_pair) > 5000:
        ts_pair = ts_pair.sample(n=5000, random_state=RANDOM_SEED)
    try:
        ts = stats.theilslopes(ts_pair[y_col].to_numpy(float), ts_pair[x_col].to_numpy(float))
        base.update({"theilsen_slope": ts.slope, "theilsen_intercept": ts.intercept, "theilsen_n": len(ts_pair)})
    except Exception:
        base.update({"theilsen_slope": np.nan, "theilsen_intercept": np.nan, "theilsen_n": len(ts_pair)})
    return base


regression_groups = {
    "all_valid_improve": valid,
    "rt_plus_loading_available": qc,
    "etad_loading_matched": loading_matched,
    "etad_loading_and_fabs": both_matched,
    "smoke_like_score_ge8": score_ge8,
    "rt_saturated": qc[qc["rt_saturated"]],
    "rt_not_saturated": qc[~qc["rt_saturated"]],
}
reg_rows = []
for name, g in regression_groups.items():
    row = {"group": name, "site_count": g["SiteCode"].nunique() if len(g) else 0}
    row.update(robust_regression_stats(g))
    row["fAbs_median"] = g["fAbs_Val"].median() if len(g) else np.nan
    row["EC_median"] = g["ECf_Val"].median() if len(g) else np.nan
    row["rt_saturated_pct"] = g["rt_saturated"].mean() * 100 if "rt_saturated" in g and len(g) else np.nan
    reg_rows.append(row)
reg_summary = pd.DataFrame(reg_rows)
reg_summary.to_csv(OUT_DIR / "improve_addis_analog_regression_sensitivity.csv", index=False)
display(reg_summary.round(4))

fig, ax = plt.subplots(figsize=(12.8, 5.8))
plot_reg = reg_summary.set_index("group")[["Slope", "origin_mac", "theilsen_slope"]]
plot_reg.plot(kind="bar", ax=ax, width=0.76)
ax.axhline(MAC_VALUE, color="0.20", lw=1.2, ls=":", label=f"MAC={MAC_VALUE:g}")
ax.set_title("IMPROVE regression sensitivity")
ax.set_ylabel("fAbs per EC slope / MAC proxy")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "improve_addis_analog_regression_sensitivity.png", dpi=240, bbox_inches="tight")
plt.show()

### Interpretation

The key regression result is the ETAD-loading-matched subset:

- it contains thousands of samples,
- its OLS fit is not Addis-like in the same way,
- its origin MAC proxy is near the expected single-digit range,
- the very-high-fAbs subset is too small and event-biased to define a stable instrument floor.

The smoke-like and saturated subsets have high intercept-like behavior, but those groups are selected on high fAbs/optical loading by construction. They are event diagnostics, not proof of a persistent Addis-style baseline offset.

## 8. Paper-ready answer

### Best-supported interpretation from IMPROVE

The IMPROVE network weakens the hypothesis that high EC surface loading alone explains the Addis HIPS anomaly. IMPROVE contains many rows in the Addis/ETAD EC surface-loading regime, but only a very small number also occupy the Addis/ETAD fAbs range. The loading-matched IMPROVE relationship remains broadly normal compared with the persistent Addis optical floor.

### What IMPROVE supports

| Finding | Interpretation |
|---|---|
| Thousands of IMPROVE rows match Addis/ETAD EC surface loading | The loading regime is not unique enough by itself to force an Addis-like offset |
| Only dozens match both Addis/ETAD loading and fAbs | True analogs are rare |
| High-score candidates cluster by site/month and have smoke-like chemistry | Many high-fAbs rows are likely episodic smoke/event samples |
| RT saturation is elevated among high-score candidates | Some high-fAbs rows are also optical-regime edge cases |
| IMPROVE export lacks blank rows and a0/a1 coefficients | It cannot repeat the Addis SOP blank-zeroing audit |

### What IMPROVE rules out or weakens

- **Generic high EC surface loading** as a sufficient explanation.
- **Network-wide fAbs-vs-EC behavior** that naturally mimics the Addis intercept.
- **Treating all high-fAbs IMPROVE rows as clean Addis analogs** without smoke and RT-saturation screening.

### What IMPROVE does not rule out

- Addis-specific HIPS/PTFE sample-side response.
- Addis-specific raw HIPS T/R geometry.
- Field-blank or coefficient issues, because the necessary blank/coefficient fields are not present in this IMPROVE export.
- True heavy-smoke analogs, because high-score IMPROVE candidates should be checked against HMS/FIRMS or equivalent fire products.

### Extra data needed for root-cause confirmation

1. Raw IMPROVE HIPS `T`/`R` and blank rows for the candidate samples.
2. Lot-level `a0`, `a1` coefficients and PTFE lot metadata.
3. HIPS processing/QC flags for saturation or out-of-range verification.
4. Potassium or biomass-burning tracer fields if available from IMPROVE/FED.
5. Fire/smoke context for candidate dates/sites.

### Bottom line

IMPROVE is useful as a stress test: it shows that high absorption events exist, but they are rare and event-like after matching Addis/ETAD loading. That makes the Addis anomaly look less like a universal high-loading behavior and more like a site/sample-side HIPS response problem that needs raw HIPS and blank-coefficient records for confirmation.

## 9. Exports

In [ ]:
export_paths = [
    OUT_DIR / "spartan_rebuilt_reference.csv",
    OUT_DIR / "improve_joined_audit_qc.csv",
    OUT_DIR / "improve_analog_group_counts.csv",
    OUT_DIR / "improve_analog_fit_summary.csv",
    OUT_DIR / "improve_smoke_candidate_summary.csv",
    OUT_DIR / "improve_smoke_event_candidates.csv",
    OUT_DIR / "improve_smoke_candidate_site_month_counts.csv",
    OUT_DIR / "improve_rt_saturation_candidates.csv",
    OUT_DIR / "improve_rt_saturation_summary.csv",
    OUT_DIR / "improve_addis_analog_regression_sensitivity.csv",
    OUT_DIR / "spartan_reference_baseline.png",
    OUT_DIR / "improve_addis_analog_overview.png",
    OUT_DIR / "improve_smoke_candidate_context.png",
    OUT_DIR / "improve_rt_saturation_audit.png",
    OUT_DIR / "improve_addis_analog_regression_sensitivity.png",
]
exports = pd.DataFrame([
    {"path": str(path), "exists": path.exists(), "size_mb": path.stat().st_size / 1_000_000 if path.exists() else np.nan}
    for path in export_paths
])
display(exports)

print("Audit readout")
print(f"- IMPROVE valid positive EC/fAbs rows: {len(valid):,}")
print(f"- IMPROVE RT + loading rows: {len(qc):,}")
print(f"- ETAD-loading-matched rows: {len(loading_matched):,}")
print(f"- ETAD fAbs + loading rows: {len(both_matched):,}")
print(f"- fAbs>=70 + ETAD loading rows: {len(high_tail_loading):,}")
print(f"- p99 + ETAD-low loading + OC/EC>=5 candidates: {len(p99_loading_ocec):,}")
print(f"- smoke_like_score>=8 candidates: {len(score_ge8):,}")
print(f"- RT saturated rows in QC set: {int(qc['rt_saturated'].sum()):,}")